# Preprocessing

Use the project's `.venv` kernel. If scikit-learn is missing, run `uv pip install scikit-learn`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load the raw data.
csv_path = Path("data/raw/student-mental.csv")
if not csv_path.is_file():
    csv_path = Path("..") / csv_path

df = pd.read_csv(csv_path)
target = "Mental_Health_Risk"

assert df["Student_ID"].notna().all(), "Missing student ID."
assert df["Student_ID"].is_unique, "Duplicate student ID."
assert df[target].isin(["Low", "Medium", "High"]).all(), "Invalid risk label."

In [ ]:
# Keep all rows and use the same class proportions in both sets.
X = df.drop(columns=["Student_ID", target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

# Fill missing numbers with the median, then scale them.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Fill missing labels, then turn categories into numbers.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
]).set_output(transform="pandas")

In [ ]:
# Learn from the training data and use the same settings on the test data.
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

X_train_ready.shape, X_test_ready.shape